# 02 - 관리자로 레지스트리 생성

이 Notebook에서는 관리자 페르소나를 사용하여 새 AWS Agent Registry를 생성하는 과정을 살펴보고, Registry와 관련된 다양한 API 작업을 설명합니다.

## 학습 내용

- 관리자 페르소나를 사용하여 **Registry** 생성 및 구성
- 계정의 레지스트리 나열
- 레지스트리 구성 업데이트

## 사전 요구 사항

- boto3 >= 1.42.87
- 관리자, 게시자, 소비자 페르소나용 IAM 역할을 생성하려면 [Notebook 01](01-create-user-personas-workflow.ipynb)을 실행하세요.

## 관리자 API 참조

| # | API | 설명 |
|---|-----|-------------|
| 1 | [CreateRegistry](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/create_registry.html) | 승인 구성을 사용하는 새 레지스트리 생성 |
| 2 | [GetRegistry](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/get_registry.html) | 레지스트리 세부 정보를 가져오고 READY 상태가 될 때까지 폴링 |
| 3 | [ListRegistries](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/list_registries.html) | 계정의 모든 레지스트리 나열 |
| 4 | [UpdateRegistry](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/update_registry.html) | 레지스트리 설명 또는 승인 구성 업데이트 |
| 5 | [DeleteRegistry](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/delete_registry.html) | ID로 레지스트리 삭제 |
| 6 | [ListRegistryRecords](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/list_registry_records.html) | 레지스트리의 모든 레코드 나열(정리) |
| 7 | [DeleteRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/delete_registry_record.html) | 레지스트리에서 특정 레코드 삭제(정리) |

#### Notebook 진행 순서

**02(이 Notebook)** → 03(레코드 게시) → 04(관리자 승인) → 05(시맨틱 검색)

#### 사용 사례: 엔터프라이즈 결제 처리
**관리자 페르소나:** AnyCompany의 관리자는 결제 처리 에이전트와 도구를 위한 중앙 레지스트리를 생성합니다. 이를 통해 고객 서비스 AI 에이전트는 배포마다 별도의 통합을 구축할 필요 없이 표준화된 결제, 환불, 거래 기능을 검색하고 사용할 수 있습니다.

---
## 1. boto3 SDK 및 종속성 설치

핵심 종속성(`boto3` 및 `python-dotenv`)을 설치합니다.

In [ ]:
!pip install boto3 python-dotenv --force-reinstall

## 2. 관리자로 boto3 Session 초기화

`admin_persona` IAM 역할을 수임하고 임시 자격 증명으로 boto3 Session을 생성합니다. 이후의 모든 API 호출은 이 Session을 사용합니다.

In [ ]:
import boto3
import time
import utils
import os

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

# 현재 자격 증명에서 계정 ID 자동 감지
sts = boto3.client("sts", region_name=AWS_REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]
CALLER_ARN = sts.get_caller_identity()["Arn"]

ADMIN_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/admin_persona"

print(f"Account:  {ADMIN_ROLE_ARN}")

# 관리자 역할 수임
creds = utils.assume_role(
    role_arn=ADMIN_ROLE_ARN,
    session_name="admin-session",
)

admin_session = boto3.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=AWS_REGION,
)

## 3. Control Plane 클라이언트 초기화

Control plane(`bedrock-agentcore-control`)은 레지스트리와 레코드에 대한 CRUD 작업을 처리합니다.

In [ ]:
# Control plane 클라이언트(관리자 작업)
cp_client = admin_session.client("bedrock-agentcore-control")

---
## 4. 레지스트리 생성

관리자는 게시자가 레코드를 제출할 레지스트리를 생성할 수 있습니다.

`autoApproval: False`가 기본값입니다. 레코드가 검색 가능해지려면 관리자의 명시적인 승인이 필요합니다.

In [ ]:
NEW_REGISTRY_NAME = "AWSAgentRegistry"  # 필요에 맞게 변경
NEW_REGISTRY_DESCRIPTION = "Registry created during the getting-started workshop"  # 필요에 맞게 변경

print(f"This will create registry: {NEW_REGISTRY_NAME}\n")

resp = cp_client.create_registry(
    name=NEW_REGISTRY_NAME,
    description=NEW_REGISTRY_DESCRIPTION,
    approvalConfiguration={"autoApproval": False},
)

REGISTRY_ARN = resp["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]

print(f"Created registry: {NEW_REGISTRY_NAME} (ID: {REGISTRY_ID})")

### 4.1 레지스트리가 준비될 때까지 대기

레지스트리 생성은 비동기 작업입니다. 상태가 `CREATING`에서 `READY`로 전환될 때까지 `GetRegistry`를 폴링합니다.

In [ ]:
while True:
    r = cp_client.get_registry(registryId=REGISTRY_ID)
    if r["status"] == "READY":
        print("Registry is READY")
        break
    print(f"Status: {r['status']} - waiting...")
    time.sleep(3)

### 4.2 레지스트리 세부 정보 확인

전체 레지스트리 객체를 가져와 이름, 설명, 승인 구성, 타임스탬프를 확인합니다.

In [ ]:
registry = cp_client.get_registry(registryId=REGISTRY_ID)
utils.pp(registry)


## 5. 레지스트리 나열

계정의 모든 레지스트리를 나열합니다. 상태(`CREATING` 또는 `READY`)로 필터링할 수 있습니다.

In [ ]:
# 모든 레지스트리 나열
registries = cp_client.list_registries()
print(f"Found {len(registries.get('registries', []))} registries:\n")
print(f"{'#':<4} {'Name':<30} {'Registry ID':<20} {'Status':<18} {'Created At':<28} {'Updated At'}")
print("-" * 140)
for i, reg in enumerate(registries.get("registries", []), 1):
    print(
        f"{i:<4} {reg['name']:<30} {reg['registryId']:<20} {reg['status']:<18} {str(reg.get('createdAt', 'N/A')):<28} {str(reg.get('updatedAt', 'N/A'))}"
    )


## 6. 레지스트리 구성 업데이트

다음 예제에서는 Registry 설명을 변경하는 방법을 보여 줍니다.

In [ ]:
updated = cp_client.update_registry(
    registryId=REGISTRY_ID,
    description={"optionalValue": "Registry created and updated"},
)

print(f"Registry entry updated with New description: {updated['description']}")
print(f"Updated at: {updated['updatedAt']}")

registry = cp_client.get_registry(registryId=REGISTRY_ID)
utils.pp(registry)

### 6.1 업데이트 확인

레지스트리를 다시 가져와 설명 변경 사항이 적용되었는지 확인합니다.

In [ ]:
registry = cp_client.get_registry(registryId=REGISTRY_ID)
utils.pp(registry)

---
## 7. 레지스트리 및 레코드 삭제(주의하여 실행)

레지스트리가 최종 상태 또는 안정된 상태일 때 삭제할 수 있습니다.

`CREATING`, `UPDATING`, `DELETING`과 같은 전환 상태에서는 삭제하지 마세요. 서비스가 아직 처리 중이므로 삭제 호출이 실패하거나 충돌할 수 있습니다.

| 상태 | 유형 | 설명 |
|--------|------|-------------|
| `CREATING` | 전환 상태 | 레지스트리를 프로비저닝하는 중 |
| `READY` | 최종 상태 | 레지스트리가 활성 상태이며 사용 가능 |
| `UPDATING` | 전환 상태 | 레지스트리 업데이트 진행 중 |
| `DELETING` | 전환 상태 | 레지스트리 삭제 진행 중 |
| `CREATE_FAILED` | 최종 상태 | 레지스트리 프로비저닝 실패 |
| `UPDATE_FAILED` | 최종 상태 | 레지스트리 업데이트 실패 |
| `DELETE_FAILED` | 최종 상태 | 레지스트리 삭제 실패 |

⚠️ 레지스트리의 모든 레코드를 삭제한 다음 레지스트리 자체를 삭제하려면 아래 코드의 주석을 해제하세요.

In [ ]:
# import botocore.exceptions

# REGISTRY_ID = "xa9Ms0EuuzddjpSF" # 특정 Registry ID로 변경할 수 있습니다.

# try:
#     # 먼저 레지스트리 상태 확인
#     reg = cp_client.get_registry(registryId=REGISTRY_ID)
#     reg_status = reg.get("status", "")

#     if reg_status == "CREATE_FAILED":
#         cp_client.delete_registry(registryId=REGISTRY_ID)
#         print(f"Deleted registry in {reg_status} state.")
#     else:
#         # 레지스트리의 모든 레코드 삭제
#         records = cp_client.list_registry_records(registryId=REGISTRY_ID)
#         for rec in records["registryRecords"]:
#             cp_client.delete_registry_record(
#                 registryId=REGISTRY_ID,
#                 recordId=rec["recordId"]
#             )
#             print(f"Deleted record: {rec['recordId']}")

#         # 레지스트리 삭제
#         cp_client.delete_registry(registryId=REGISTRY_ID)
#         print(f"Deleted registry: {REGISTRY_ID}")

#     print("\nCleanup complete!")

# except cp_client.exceptions.ResourceNotFoundException:
#     print(f"Registry {REGISTRY_ID} not found - already cleaned up.")
# except botocore.exceptions.ClientError as e:
#     print(f"Error during cleanup: {e}")

# # 모든 레지스트리 나열
# registries = cp_client.list_registries()
# print(f"Found {len(registries.get('registries', []))} registries:\n")
# print(f"{'#':<4} {'Name':<30} {'Registry ID':<20} {'Status':<18} {'Created At':<28} {'Updated At'}")
# print("-" * 140)
# for i, reg in enumerate(registries.get('registries', []), 1):
#     print(f"{i:<4} {reg['name']:<30} {reg['registryId']:<20} {reg['status']:<18} {str(reg.get('createdAt','N/A')):<28} {str(reg.get('updatedAt','N/A'))}")

## 사전 요구 Notebook
- **Notebook 01** — [사용자 페르소나 생성](01-create-user-personas-workflow.ipynb): 관리자, 게시자, 소비자 사용자 페르소나 설정

## 다음 단계
- **Notebook 03** — [레코드 게시](03-publishing-records-workflow.ipynb): 게시자로 레코드 게시
- **Notebook 04** — [관리자 승인](04-admin-approval-workflow.ipynb): 관리자 승인 워크플로
- **Notebook 05** — [시맨틱 검색](05-search-registry-workflow.ipynb): 소비자로서 NLQ를 사용해 승인된 레코드 검색